# 6.2 简单 CNN：log-mel谱 + 数据增强

基于CTMP清洗数据集（6类、24 kHz、5秒切片），用 log-mel 谱和 3 层 CNN 训练端到端分类器，对比“无增强”与“全套增强”两种配置。

评估协议（与 6.1b 一致）：
- **选择**：train split 训练 → val split 调整学习率
- **内部**：训练完成后在 test split 评估
- **外部**：同一模型直接在 external_test（ChMusic）上推理
- **划分敏感性**：在 3 份冻结的 train/val/test 划分上评估，取均值和总体标准差；external_test固定不变

> 数据来源：`_data_pipeline/output/`。若未跑过清洗流程，请先按
> `06_1a_dataset_exploration.ipynb` §9.0 的步骤生成。


## 1. 环境准备


In [ ]:
import sys
from pathlib import Path

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录；PROJECT_ROOT 指向 CODE/
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
PROJECT_ROOT = _p / "CODE"  # CODE/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = [
    'Hiragino Sans GB', 'PingFang SC', 'Arial Unicode MS', 'STHeiti', 'Heiti TC',
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans',
]
matplotlib.rcParams['axes.unicode_minus'] = False
print('matplotlib:', matplotlib.__version__)


### 1.1 依赖与数据齐备性检查

下面的单元统一检查PyTorch、librosa和CTMP切片；若依赖缺失，检查会立即报错并列出相应的安装命令。


In [ ]:
from chapter06._common import check_environment

check_environment(notebook='06_2', require_ctmp=True)


In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

from chapter06._common import (
    add_recall_colorbar,
    get_device,
    plot_confusion_matrix,
    setup_chinese_font,
)
from chapter06._common.ctmp_loader import (
    CTMP_CLASSES,
    build_label_map,
    get_ctmp_output_dir,
    load_ctmp_segments,
)
from chapter06.simple_cnn import (
    AugmentConfig,
    SimpleAudioCNN,
    TrainConfig,
    run_experiment,
)

setup_chinese_font()

OUT_DIR = PROJECT_ROOT / 'chapter06' / 'simple_cnn' / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [0, 1, 2]
# 变量名沿用代码接口；0/1/2 实际表示三份冻结 manifest 的编号。
LABEL_MAP = build_label_map()
CLASS_NAMES = list(CTMP_CLASSES)
N_CLASSES = len(CLASS_NAMES)

print(f'CTMP 类别: {CLASS_NAMES} ({N_CLASSES} 类)')
print(f'CTMP 输出目录: {get_ctmp_output_dir()}')
print(f'device: {get_device()}')
print(f'torch: {torch.__version__}')


## 2. log-mel谱可视化

从冻结划分0的train和external_test中为每个类别各取一段切片，绘制log-mel谱。
这些样例可用于查看谱图外观上的可能差异，但不能代表各数据源的整体分布，也不能据此确定差异成因。


In [ ]:
import soundfile as sf
from chapter06.simple_cnn.dataset import compute_logmel, SAMPLE_RATE

train_segs = load_ctmp_segments(seed=0, split='train')
ext_segs = load_ctmp_segments(seed=0, split='external_test')

fig, axes = plt.subplots(2, N_CLASSES, figsize=(20, 6), sharey=True)
for j, cls in enumerate(CLASS_NAMES):
    # 内部 train
    seg = next(s for s in train_segs if s['family_label'] == cls)
    audio, sr = sf.read(seg['audio_path'], dtype='float32')
    spec = compute_logmel(audio, sr)
    axes[0, j].imshow(spec, origin='lower', aspect='auto', cmap='gray_r')
    axes[0, j].set_title(f'内部 · {cls}')
    if j == 0:
        axes[0, j].set_ylabel('mel bin')

    # 外部 external_test
    seg_ext = next(s for s in ext_segs if s['family_label'] == cls)
    audio_ext, sr_ext = sf.read(seg_ext['audio_path'], dtype='float32')
    spec_ext = compute_logmel(audio_ext, sr_ext)
    axes[1, j].imshow(spec_ext, origin='lower', aspect='auto', cmap='gray_r')
    axes[1, j].set_title(f'外部 · {cls}')
    if j == 0:
        axes[1, j].set_ylabel('mel bin')
    axes[1, j].set_xlabel('帧')

fig.suptitle('log-mel谱对比：内部train（上）与外部external_test（下）', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'logmel_comparison.png', dpi=600, bbox_inches='tight')
plt.show()


## 3. 模型与增强配置

- `SimpleAudioCNN`：3 个 conv block（32→64→128 通道）+ GroupNorm + ReLU + MaxPool
  + GAP + FC。GroupNorm 在每个样本内部按通道组归一化，不依赖小 batch 的批维统计量。
- 增强分三层：波形层（加噪、音量抖动、pitch shift、time stretch）、频谱层
  （SpecAugment）、batch 层（Mixup）。
- Label smoothing（ε=0.1）+ SWA（后 25% epoch 等权重平均）：前者软化目标分布，
  后者对训练后段的参数做平均。它们是否改善本任务的外部表现需要单独消融验证。


In [ ]:
model_preview = SimpleAudioCNN(n_classes=N_CLASSES)
print(f'参数量: {model_preview.num_params():,}')
print(model_preview)


In [ ]:
aug_off = AugmentConfig(enabled=False)
aug_on = AugmentConfig(
    enabled=True,
    noise_std=0.003,
    gain_db_range=(-3.0, 3.0),
    pitch_semitones=1.0,
    time_stretch_range=(0.95, 1.05),
    aug_prob=0.3,
    n_time_masks=2, time_mask_max=10,
    n_freq_masks=2, freq_mask_max=8,
    spec_aug_prob=0.4,
    mixup_alpha=0.2,
)
print('无增强:', aug_off)
print('全套增强:', aug_on)


## 4. 训练与评估（3 份冻结划分 × 2 配置）

每份划分：在 train 上训练，用 val 调整学习率；训练完成后才在 test（内部）
和 external_test（外部）上分别评估。
两种配置：无增强、全套增强。


In [ ]:
train_cfg = TrainConfig(
    epochs=60, batch_size=16, lr=3e-3, weight_decay=1e-4,
    num_workers=0, log_every=10, label_smoothing=0.1,
    # SWA 默认值：swa_start_frac=0.75（第 45 epoch 开始），swa_lr=1e-3
)

all_results = []
RUN_CACHE = OUT_DIR / 'run_cache' / 'simple_cnn_val'

for aug_name, aug_cfg in [('无增强', aug_off), ('全套增强', aug_on)]:
    for seed in SEEDS:
        t0 = time.perf_counter()
        result = run_experiment(
            seed=seed, aug=aug_cfg, cfg=train_cfg,
            tag=f'{aug_name}-split{seed}',
            cache_dir=RUN_CACHE,
            cache_key='no_aug' if aug_name == '无增强' else 'all_aug',
        )
        result['aug_name'] = aug_name
        all_results.append(result)
        print(f'  [{aug_name}] split {seed} 完成，耗时 {time.perf_counter() - t0:.1f}s')
    print()


## 5. 主结果表：3 份冻结划分的均值 ± 总体标准差

内部（test）与外部（external_test）的准确率和 macro-F1 并列，Gap = 内部 − 外部。


In [ ]:
summary_rows = []
for aug_name in ['无增强', '全套增强']:
    sub = [r for r in all_results if r['aug_name'] == aug_name]
    test_accs = [r['test_acc'] for r in sub]
    test_f1s = [r['test_f1'] for r in sub]
    ext_accs = [r['ext_acc'] for r in sub]
    ext_f1s = [r['ext_f1'] for r in sub]
    summary_rows.append({
        '方法': f'CNN · {aug_name}',
        '内部 Acc': f"{np.mean(test_accs):.3f} ± {np.std(test_accs):.3f}",
        '内部 F1': f"{np.mean(test_f1s):.3f} ± {np.std(test_f1s):.3f}",
        '外部 Acc': f"{np.mean(ext_accs):.3f} ± {np.std(ext_accs):.3f}",
        '外部 F1': f"{np.mean(ext_f1s):.3f} ± {np.std(ext_f1s):.3f}",
        'Gap (Acc)': f"{np.mean(test_accs) - np.mean(ext_accs):+.3f}",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'method_comparison.csv', index=False)
print('written:', (OUT_DIR / 'method_comparison.csv').resolve())
df_summary


## 6. 训练曲线（冻结划分0）

取冻结划分0的训练历史画loss和val指标曲线。左列无增强、右列全套增强。


In [ ]:
def plot_history(history, title, ax_loss, ax_metric):
    epochs = range(1, len(history['train_loss']) + 1)
    ax_loss.plot(epochs, history['train_loss'], label='train', color='0.3')
    ax_loss.plot(epochs, history['val_loss'], label='val', color='0.6', linestyle='--')
    ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('loss')
    ax_loss.set_title(f'{title} · loss'); ax_loss.legend()
    ax_metric.plot(epochs, history['val_acc'], label='accuracy', color='0.3')
    ax_metric.plot(epochs, history['val_f1'], label='macro-F1', color='0.6', linestyle='--')
    ax_metric.set_xlabel('epoch'); ax_metric.set_ylabel('指标')
    ax_metric.set_title(f'{title} · val 指标'); ax_metric.legend()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
seed0_results = [r for r in all_results if r['seed'] == 0]
for row, r in enumerate(seed0_results):
    plot_history(r['history'], r['aug_name'], axes[row, 0], axes[row, 1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'training_curves.png', dpi=600, bbox_inches='tight')
plt.show()


## 7. 混淆矩阵：内部与外部（冻结划分0，两种配置）

上排 = 无增强，下排 = 全套增强。左列 = test（内部），右列 = external_test（外部）。


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for row, r in enumerate(seed0_results):
    plot_confusion_matrix(
        axes[row, 0], r['y_test'], r['pred_test'],
        class_names=CLASS_NAMES, title=f'{r["aug_name"]} — 内部 test',
        labels=list(range(N_CLASSES)),
    )
    plot_confusion_matrix(
        axes[row, 1], r['y_ext'], r['pred_ext'],
        class_names=CLASS_NAMES, title=f'{r["aug_name"]} — 外部 external_test',
        labels=list(range(N_CLASSES)),
    )
fig.subplots_adjust(wspace=0, hspace=0.23)
add_recall_colorbar(fig, axes.ravel().tolist())
fig.savefig(FIG_DIR / 'confusion_internal_vs_external.png', dpi=600, bbox_inches='tight')
plt.show()


## 8. 与 6.1b 传统 ML 对比

读取 6.1b 的 `method_comparison.csv`，与本节 CNN 结果并列。


In [ ]:
# 读 6.1b 结果
ml_csv = PROJECT_ROOT / 'chapter06' / 'traditional_ml' / 'outputs' / 'method_comparison.csv'
if not ml_csv.exists():
    raise FileNotFoundError(
        '缺少6.1b结果：请先执行06_1b_traditional_ml.ipynb，再生成全章比较表'
    )
df_ml = pd.read_csv(ml_csv, dtype=str)
df_all = pd.concat([df_ml, df_summary], ignore_index=True)

df_all.to_csv(OUT_DIR / 'full_comparison.csv', index=False)
df_all


## 9. 结论

- 本节比较同一 CNN 在无增强与全套增强两种配置下的内部和外部结果。
- 增强配置对外部测试的影响以本次重新生成的结果表为准；两种配置只构成组合消融，
  不能分离某一种增强操作的贡献。
- 内外 gap 可由录音条件、曲目、演奏方式、样本选择等多种差异共同造成；当前元数据
  和实验设计不足以确定各因素的贡献。

**下一步（6.3）**：在同一数据划分和评估协议下，比较 AudioSet 预训练 CNN14
的三种迁移策略与本节从随机参数训练的 CNN。
